In [11]:
import pandas as pd
import numpy as np

# from sklearn.datasets import make_friedman1

from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LinearRegression
# from sklearn.ensemble import GradientBoostingRegressor
# from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

# import argparse
import sys

# import time
# import datetime

# import warnings
# warnings.filterwarnings('ignore')

sys.path.append("../")
from bblr.bblr2 import *
from bblr.simulations_functions import *

In [15]:
# import data from a file

# define variables to import the data - change manually
test_size = 0.5
file_path = f'../data/California_Housing_CitiesAdded.csv'
target = "Median_House_Value"

# data transformations
df = pd.read_csv(file_path)
columns_list = df.columns.tolist()
y_df = df[target]
df.drop(columns=target, inplace=True)
columns_list.remove(target)

X = df.to_numpy()
y = y_df.to_numpy()

# California:
X = X[:, 0:13]

In [ ]:
# generate data

# define the parameters of the generated dataset - change manually
n=1000
dgp_name="BB"
d=2
noise_var=0.2
x_distribution='normal'
test_size=0.5
random_seed = 42

# data transformations
df = generate_data(n, dgp_name, d=d, noise_var=noise_var, seed=random_seed, x_distribution=x_distribution)
columns_list = [f'X{i+1}' for i in range(d)]  # define column names
X = df[columns_list]
y = df['y']
X = X.to_numpy()
y = y.to_numpy()

In [16]:
# transform data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

# uncomment following lines to add scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
# define method and its parameters
method="BBLR"
min_pts=15
eps_range=(0.1,3.0)
eps_multiplier=0.1
bblr_outliers_method="up_to_N"
coef_comp='mean'
bblr_regression_type="OLS"
bblr_outliers_method_pred="up_to_N"

param_min = 0.5
param_max = 1.0
param_increment = 0.1

params = np.round(np.arange(param_min, param_max, param_increment), 2)


# define what output do we want to be saved to the files 
include_training_data=True
ci=True
confidence=0.95


# perform the regression
for param in params:
    print(param)
    results_df = estimate_coefs(X_train, X_test, y_train, y_test, method,
                                n=n, dgp_name=dgp_name, param=param, column_names=columns_list, min_points=min_pts,  
                                bblr_regression_type = bblr_regression_type, eps_multiplier = eps_multiplier,
                                coef_comp = coef_comp, ci=ci, pred_outliers_method=bblr_outliers_method_pred,
                                confidence = confidence, bblr_outliers_method=bblr_outliers_method,
                                eps_range = eps_range)
    save_raw_results(results_df, n, method, dgp_name, x_distribution, 'test',
                        bblr_outliers_method=bblr_outliers_method,
                        coef_comp=coef_comp,
                        bblr_regression_type=bblr_regression_type, 
                        ci=ci)
    if include_training_data:
        results_df = estimate_coefs(X_train, X_train, y_train, y_train, method,
                                    n=n, dgp_name=dgp_name, param=param, column_names=columns_list, min_points=min_pts,  
                                    bblr_regression_type = bblr_regression_type, eps_multiplier = eps_multiplier,
                                    coef_comp = coef_comp, ci=ci, pred_outliers_method=bblr_outliers_method_pred,
                                    confidence = confidence, bblr_outliers_method=bblr_outliers_method,
                                    eps_range = eps_range)
        save_raw_results(results_df, n, method, dgp_name, x_distribution, 'train',
                            bblr_outliers_method=bblr_outliers_method,
                            coef_comp=coef_comp,
                            bblr_regression_type=bblr_regression_type, 
                            ci=ci)

0.5


ValueError: Design matrix is singular. Cannot compute standard errors.